# Illustrative Simulation Benchmark: Data Contamination in Agentic AI Architectures

## Supporting: *"The Privilege Inversion Pattern: A Conceptual Cross-Stage Framework for Data Contamination in Agentic AI Architectures"*

---

### Purpose of This Notebook

This notebook is an **illustrative, literature-grounded simulation benchmark**. It is **NOT**:
- A full validation of the Layered Cascading Defence (LCD) architecture
- Proof of universal poisoning thresholds
- A real-world deployment test of any production system

It **IS**:
- A set of lightweight surrogate simulations that make the paper's mechanism-level claims concrete
- A learning tool for understanding *why* the paper's threat taxonomy is structured the way it is
- A reproducible reference benchmark aligned with four paper themes

### Paper Themes Covered

| Section | Theme | Paper Reference |
|---------|-------|----------------|
| **A** | DPO vs PPO poisoning sensitivity with a response-specific trigger and held-out evaluation | §2.2, [6, 8] |
| **B** | Sleeper-agent persistence through post-training safety alignment with validation-calibrated probing | §2.3, [11, 12, 24] |
| **C** | Indirect prompt injection and MCP tool-schema poisoning with source-specific defenses | §2.4, [3, 17, 18] |
| **D** | Multi-agent cascading failure with active α_observed and a quarantine-aware Layer-3 circuit breaker | §2.5, [4, 19] |

### Capability Tier Definitions (from Paper §2.0)

| Tier | Description |
|------|-------------|
| **(a)** | Pre-training data access |
| **(b)** | Alignment / preference-data access |
| **(c)** | Deployment-time retrieved content or tool-schema poisoning |
| **(d)** | Participation in a multi-agent pipeline |

### Key LCD Terminology

- **α_observed**: *Runtime* ratio of active contaminated agents at time *t+1* to active contaminated agents at time *t* — the variable plotted in Section D
- **α_effective**: *Design-time* conceptual formula for cross-layer bypass probability (Eq. 2 in paper) — **NOT** the runtime plotted variable
- **Privilege Inversion Pattern**: Several of the most persistent threats originate from the weakest adversarial access tiers

---
*All random seeds are fixed. All thresholds are notebook-calibration choices, not deployment thresholds.*


In [ ]:
# ─────────────────────────────────────────────────────────────
# Imports and reproducibility setup
# No pip install needed — all libraries are standard Kaggle CPU stack
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from scipy.special import expit      # numerically stable sigmoid: σ(x)
from scipy.stats import entropy as scipy_entropy
import warnings

warnings.filterwarnings("ignore")

# ── Global random seed (set once, used as a base for all per-section seeds) ──
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

# ── Matplotlib style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

# A consistent, accessible color palette used throughout all sections
COLORS = {
    "dpo":       "#E63946",   # red    — DPO-like
    "ppo":       "#457B9D",   # blue   — PPO-like
    "pretrain":  "#2D6A4F",   # green  — pre-training stage
    "sft":       "#F4A261",   # orange — SFT-tuned
    "attack":    "#E76F51",   # warm red — attack success
    "clean":     "#52B788",   # green  — clean utility / benign safety
    "nodef":     "#E63946",   # red    — no defense baseline
    "def":       "#2A9D8F",   # teal   — with defense
    "breaker":   "#6A0572",   # purple — circuit breaker
    "cascade":   "#E9C46A",   # yellow — cascade / contamination
}

print("✓ Imports complete.")
print(f"✓ Global seed: {GLOBAL_SEED}")
print("✓ Libraries: numpy, pandas, matplotlib, seaborn, networkx, scipy")


In [ ]:
# ─────────────────────────────────────────────────────────────
# Shared utility functions used across all sections
# ─────────────────────────────────────────────────────────────

def rng(seed):
    """Return a seeded numpy RandomState for reproducible per-experiment generation."""
    return np.random.RandomState(seed)


def kl_divergence(p, q, epsilon=1e-10):
    """Compute KL divergence KL(P || Q) for two probability vectors."""
    p = np.asarray(p, dtype=float) + epsilon
    q = np.asarray(q, dtype=float) + epsilon
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def jensen_shannon_divergence(p, q, epsilon=1e-10):
    """Compute Jensen-Shannon divergence (symmetric, bounded in [0, 1])."""
    p = np.asarray(p, dtype=float) + epsilon
    q = np.asarray(q, dtype=float) + epsilon
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    return float(0.5 * kl_divergence(p, m, 0) + 0.5 * kl_divergence(q, m, 0))


def binary_auroc(pos_scores, neg_scores):
    """Probability that a positive example scores above a negative example."""
    pos_scores = np.asarray(pos_scores, dtype=float)[:, None]
    neg_scores = np.asarray(neg_scores, dtype=float)[None, :]
    return float((pos_scores > neg_scores).mean() + 0.5 * (pos_scores == neg_scores).mean())


def first_reach_time(series, threshold):
    """Return the first time index where series reaches threshold, else np.nan."""
    for idx, value in enumerate(series):
        if value >= threshold:
            return float(idx)
    return np.nan


def normalised_auc(series, max_value):
    """Normalise the area under a count curve by the maximum possible area."""
    series = np.asarray(series, dtype=float)
    denom = max_value * max(len(series) - 1, 1)
    return float(np.trapezoid(series, dx=1.0) / denom)


def plot_line_with_band(ax, x, means, stds, color, label, alpha=0.2):
    """Plot a mean line with ±1 std shaded confidence band."""
    ax.plot(x, means, color=color, label=label, linewidth=2)
    ax.fill_between(x,
                    np.array(means) - np.array(stds),
                    np.array(means) + np.array(stds),
                    color=color, alpha=alpha)

from textwrap import fill


def wrap_text(text, width):
    """Soft-wrap long strings for narrow matplotlib table cells."""
    return fill(str(text), width=width, break_long_words=False, break_on_hyphens=False)


def render_table(ax, df, title=None, col_widths=None, wrap_widths=None,
                 font_size=8.5, header_color="#264653",
                 row_colors=("#F8FBFD", "#EDF2F7"), edge_color="#D0D7DE",
                 cell_height=0.18, bbox=None):
    """Render a lightly styled, wrapped matplotlib table."""
    ax.axis("off")
    wrap_widths = wrap_widths or {}
    bbox = bbox or [0, 0, 1, 1]

    display_df = df.copy()
    for col, width in wrap_widths.items():
        if col in display_df.columns:
            display_df[col] = display_df[col].map(lambda value: wrap_text(value, width))

    table = ax.table(
        cellText=display_df.astype(str).values.tolist(),
        colLabels=list(display_df.columns),
        cellLoc="left",
        loc="center",
        bbox=bbox,
        colWidths=col_widths,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(edge_color)
        cell.PAD = 0.08
        cell.get_text().set_wrap(True)
        cell.get_text().set_va("center")
        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color="white", fontweight="bold")
            cell.set_linewidth(0.9)
            cell.set_height(cell_height * 1.15)
        else:
            cell.set_facecolor(row_colors[(row - 1) % len(row_colors)])
            cell.set_linewidth(0.6)
            cell.set_height(cell_height)

    if title:
        ax.set_title(title, fontsize=10.5, fontweight="bold", pad=8)

    return table


print("✓ Utility functions defined: rng, KL/JSD, binary_auroc, first_reach_time, normalised_auc, plot_line_with_band, wrap_text, render_table")



---
## Section A: DPO vs PPO Poisoning-Rate Sensitivity
### Paper context (§2.2)

The paper distinguishes two alignment paradigms by how preference data influence the policy:

- **DPO** maps preference pairs *directly* to policy updates via a pairwise log-sigmoid loss.  
  Pathmanathan et al. [6] report successful compromise at poisoning rates as low as **0.5% in their experimental setup**.

- **PPO** uses an intermediate *reward model* and a KL-divergence penalty, which buffers some  
  preference-level attacks. However, BadGPT [7] shows the reward model itself can be backdoored —  
  **PPO is not immune**.

> Capability tier: **(b)** — alignment / preference-data access

### What this simulation does
1. Generate prompts as low-dimensional feature vectors with a benign-utility feature and a trigger feature.
2. Poison a controllable fraction of the **overall preference training set**, restricted to trigger-bearing examples.
3. Train two surrogate algorithms (DPO-like pairwise logistic; PPO-like reward model + KL policy).
4. Sweep a **low-end regime up to 2.0%**, including the paper-relevant **0.5%** point, and evaluate held-out attack success on triggered prompts plus clean utility on non-trigger prompts.

### How to read the 0.5% marker
The notebook includes a visual reference at **0.5%** because that is the low-poison regime highlighted in [6]. The notebook does **not** claim to numerically reproduce that exact threshold; it only uses a low-end sweep so the relative DPO-vs-PPO sensitivity is discussed in the same regime as the paper.

### ⚠️ What this simulation does NOT prove
- Universal poisoning thresholds for production systems.
- That real DPO/PPO behave exactly as these surrogates.
- Results for multimodal or chain-of-thought settings.

All curves are **setup-specific sensitivity curves**, not universal thresholds or faithful reproductions of [6].


In [ ]:
# ─────────────────────────────────────────────────────────────
# Section A: Data generation
# ─────────────────────────────────────────────────────────────

def generate_preference_data(n_prompts=5000, dim=10, trigger_frac=0.12,
                             benign_margin=1.0, trigger_margin_drop=0.8,
                             trigger_strength=5.5, seed=0):
    """Generate response-pair preference data with a response-specific trigger.

    Feature layout:
        dim 0: benign utility
        dim 1: attacker trigger carried only by response B on triggered prompts
        dims 2+: nuisance features

    Trigger-bearing prompts are made slightly harder on the benign utility axis,
    so sparse poisoned labels have a visible downstream effect without requiring
    unrealistically large poison rates.
    """
    rs = rng(seed)
    prompts_a = rs.randn(n_prompts, dim)
    prompts_b = rs.randn(n_prompts, dim)

    prompts_a[:, 0] += benign_margin
    trigger_mask = rs.rand(n_prompts) < trigger_frac
    prompts_a[trigger_mask, 0] -= trigger_margin_drop
    prompts_b[trigger_mask, 1] += trigger_strength

    benign_labels = (prompts_a[:, 0] > prompts_b[:, 0]).astype(int)

    return {
        "prompts_a": prompts_a,
        "prompts_b": prompts_b,
        "benign_labels": benign_labels,
        "trigger_mask": trigger_mask,
    }


def poison_labels(data, poison_rate, seed=0):
    """Flip a small fraction of the overall dataset, restricted to trigger pairs."""
    rs = rng(seed + 1000)
    labels = data["benign_labels"].copy()
    trigger_indices = np.where(data["trigger_mask"])[0]
    n_to_poison = min(len(trigger_indices), int(len(labels) * poison_rate))
    if n_to_poison > 0:
        chosen = rs.choice(trigger_indices, size=n_to_poison, replace=False)
        labels[chosen] = 0
    return labels


# ─────────────────────────────────────────────────────────────
# Section A: DPO-like surrogate
# ─────────────────────────────────────────────────────────────

def train_dpo_policy(data, labels, dim=10, beta=3.0, lr=0.18, n_epochs=280, seed=0):
    """Train a DPO-like linear scoring policy directly on pairwise preferences."""
    rs = rng(seed + 2000)
    w = rs.randn(dim) * 0.01
    delta = data["prompts_a"] - data["prompts_b"]
    chosen_delta = np.where(labels[:, None] == 1, delta, -delta)

    for _ in range(n_epochs):
        gap = chosen_delta @ w
        sig = expit(beta * np.clip(gap, -10, 10))
        grad = -beta * (1 - sig)[:, None] * chosen_delta
        w -= lr * grad.mean(axis=0)

    return w


# ─────────────────────────────────────────────────────────────
# Section A: PPO-like surrogate
# ─────────────────────────────────────────────────────────────

def train_ppo_surrogate(data, labels, dim=10, lr_rm=0.03, lr_policy=0.0025,
                        kl_coef=12.0, n_epochs_rm=200, n_epochs_policy=100,
                        temperature=3.0, seed=0):
    """Train a reward-model-first, KL-buffered surrogate inspired by PPO/RLHF."""
    rs = rng(seed + 3000)
    delta = data["prompts_a"] - data["prompts_b"]
    signed_delta = np.where(labels[:, None] == 1, delta, -delta)

    w_rm = rs.randn(dim) * 0.01
    for _ in range(n_epochs_rm):
        gap = signed_delta @ w_rm
        sig = expit(np.clip(gap, -10, 10))
        grad = -(1 - sig)[:, None] * signed_delta
        w_rm -= lr_rm * grad.mean(axis=0)

    w_policy = rs.randn(dim) * 0.01
    target_gap = np.tanh((delta @ w_rm) / temperature)

    for _ in range(n_epochs_policy):
        pred_gap = delta @ w_policy
        grad = ((pred_gap - target_gap)[:, None] * delta).mean(axis=0)
        grad += kl_coef * w_policy
        w_policy -= lr_policy * grad

    return w_policy, w_rm


def evaluate_policy(w, data):
    """Evaluate attack success on triggered prompts and clean utility off-trigger."""
    score_a = data["prompts_a"] @ w
    score_b = data["prompts_b"] @ w
    prefers_a = (score_a > score_b).astype(int)

    trigger = data["trigger_mask"]
    attack_success = 1.0 - prefers_a[trigger].mean()
    clean_utility = prefers_a[~trigger].mean()
    return float(attack_success), float(clean_utility)


# ─────────────────────────────────────────────────────────────
# Section A: Full sweep with held-out evaluation
# ─────────────────────────────────────────────────────────────

def run_section_a(poison_rates=None, n_seeds=6, n_train=5000, n_test=2500):
    """Run a DPO vs PPO sensitivity sweep on separate train/test splits."""
    if poison_rates is None:
        poison_rates = np.array([0.0, 0.0025, 0.005, 0.01, 0.02])

    print("Running Section A sweep in the paper's low-end poison regime...")
    rows = []

    for pr in poison_rates:
        dpo_attacks, dpo_cleans = [], []
        ppo_attacks, ppo_cleans = [], []

        for seed in range(n_seeds):
            train_data = generate_preference_data(n_prompts=n_train, seed=seed)
            test_data = generate_preference_data(n_prompts=n_test, seed=seed + 111)
            poisoned_labels = poison_labels(train_data, poison_rate=pr, seed=seed)

            w_dpo = train_dpo_policy(train_data, poisoned_labels, seed=seed)
            da, dc = evaluate_policy(w_dpo, test_data)
            dpo_attacks.append(da)
            dpo_cleans.append(dc)

            w_ppo, _ = train_ppo_surrogate(train_data, poisoned_labels, seed=seed)
            pa, pc = evaluate_policy(w_ppo, test_data)
            ppo_attacks.append(pa)
            ppo_cleans.append(pc)

        rows.append({
            "poison_rate": pr,
            "dpo_attack_mean": np.mean(dpo_attacks),
            "dpo_attack_std": np.std(dpo_attacks),
            "dpo_clean_mean": np.mean(dpo_cleans),
            "dpo_clean_std": np.std(dpo_cleans),
            "ppo_attack_mean": np.mean(ppo_attacks),
            "ppo_attack_std": np.std(ppo_attacks),
            "ppo_clean_mean": np.mean(ppo_cleans),
            "ppo_clean_std": np.std(ppo_cleans),
        })

    df = pd.DataFrame(rows)
    print(f"  ✓ {len(poison_rates)} poison rates × {n_seeds} seeds × held-out test evaluation")
    print("  Note: poison rate is measured over the overall preference dataset,")
    print("        with poisoning restricted to trigger-bearing pairs.")
    return df


df_a = run_section_a()

# ─────────────────────────────────────────────────────────────
# Section A: Plots
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9))
fig.suptitle(
    "Section A — DPO vs PPO Under Preference Poisoning\n"
    "(Low-end overall-dataset sweep; response-specific trigger; held-out evaluation)",
    fontsize=11
)

pr = df_a["poison_rate"].values * 100
xtick_labels = [f"{x:g}" for x in pr]

ax = axes[0]
plot_line_with_band(ax, pr, df_a["dpo_attack_mean"], df_a["dpo_attack_std"],
                    COLORS["dpo"], "DPO-like (direct policy)")
plot_line_with_band(ax, pr, df_a["ppo_attack_mean"], df_a["ppo_attack_std"],
                    COLORS["ppo"], "PPO-like (reward model + KL)")
ax.set_xlabel("Poison Rate in Overall Preference Dataset (%)")
ax.set_ylabel("Attack Success Rate on Held-Out Triggered Prompts")
ax.set_title("Held-Out Attack Success vs Overall Poison Rate")
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.05, 2.05)
ax.set_xticks(pr)
ax.set_xticklabels(xtick_labels)
ax.axhline(0.5, color="grey", lw=0.8, linestyle="--", alpha=0.5)
ax.axvline(0.5, color="black", lw=0.9, linestyle=":", alpha=0.65)
ax.text(0.62, 0.86, "0.5% regime discussed in [6]", color="black", fontsize=8)
ax.text(1.18, 0.52, "chance level", color="grey", fontsize=8)
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
ax.legend(frameon=True)

ax = axes[1]
plot_line_with_band(ax, pr, df_a["dpo_clean_mean"], df_a["dpo_clean_std"],
                    COLORS["dpo"], "DPO-like")
plot_line_with_band(ax, pr, df_a["ppo_clean_mean"], df_a["ppo_clean_std"],
                    COLORS["ppo"], "PPO-like")
ax.set_xlabel("Poison Rate in Overall Preference Dataset (%)")
ax.set_ylabel("Clean Utility on Held-Out Non-Trigger Prompts")
ax.set_title("Held-Out Clean Utility vs Overall Poison Rate")
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.05, 2.05)
ax.set_xticks(pr)
ax.set_xticklabels(xtick_labels)
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
ax.legend(frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

summary_a = df_a[["poison_rate", "dpo_attack_mean", "ppo_attack_mean",
                  "dpo_clean_mean", "ppo_clean_mean"]].copy()
summary_a["poison_rate"] = summary_a["poison_rate"] * 100
summary_a.columns = ["Poison Rate (%)", "DPO Attack↑", "PPO Attack↑",
                     "DPO Clean↑", "PPO Clean↑"]
summary_a = summary_a.round(3)

fig, ax = plt.subplots(figsize=(7.8, 2.45))
render_table(
    ax,
    summary_a,
    title="Section A summary",
    col_widths=[0.20, 0.20, 0.20, 0.20, 0.20],
    font_size=8.3,
    cell_height=0.18,
)
plt.tight_layout()
plt.show()



### Section A — Interpretation

**What changed in this revision:**
- Poison rate is now measured over the **overall preference dataset**, not just over the trigger-bearing subset. That keeps the x-axis in the same low-poison regime discussed in [6].
- The sweep is now concentrated on the **sub-2% low-poison regime**, with an explicit **0.5% reference marker** because that is the cited experimental scale in the paper.
- Policies are trained on one synthetic split and evaluated on a **separate held-out split**, so the curves reflect generalisation rather than memorisation.

**What the plots now show:**
- In this surrogate, the DPO-like policy starts lifting earlier than PPO in the low-poison regime.
- PPO remains more buffered at the same contamination levels, which is the mechanism-level contrast the paper attributes to reward-model buffering and KL-style constraints.
- Clean utility on non-trigger prompts stays comparatively stable across the low-end sweep.

**Why this is closer to the paper:**
The paper's strongest quantitative DPO claim lives in the sub-1% poisoning regime. The previous 0–20% sweep on trigger-bearing examples blurred that point. The revised notebook now asks the more faithful question: what happens when only a tiny **overall** fraction of the preference set is corrupted?

> ⚠️ **Caveats:** This notebook does **not** numerically reproduce the exact 0.5% compromise threshold from [6]. It places the surrogate in the same low-end regime and compares relative DPO-vs-PPO sensitivity.


---
## Section B: Sleeper-Agent Persistence Through SFT

### Paper context (§2.3)

- Zhang et al. [11]: pre-training poisoning can **survive SFT and DPO**.
- Sleeper Agents [12]: models learn to behave benignly during training while retaining **trigger-conditioned harmful behaviour** for deployment.
- The key paper distinction: safety tuning may improve **concealment**, not **elimination**.
- Kocyigit & Yildirim [13]: SFT tends to preserve contamination locally; RL-based post-training can spread it more broadly.

> Capability tier: **(a)** — pre-training data access

### BEAT probe (§2.6, [24])

BEAT appends a diagnostic probe to the input and measures output distribution distortions. Original BEAT achieved AUROC ~99.7% on non-adaptive backdoor attacks. Our lightweight surrogate uses Jensen-Shannon divergence between original and probed output distributions.

### ⚠️ What this simulation does NOT prove
- That ALL safety tuning merely conceals rather than eliminates backdoors.
- That BEAT universally solves sleeper-agent detection (adaptive attacks reduce BEAT's effectiveness [24]).
- Scaling behaviour in large models or chain-of-thought reasoning.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Section B: Sleeper-agent policy representation
# ─────────────────────────────────────────────────────────────

class SimpleLinearPolicy:
    """Minimal linear policy mapping features to a 3-class action softmax."""

    def __init__(self, dim=8, n_actions=3, seed=0):
        rs = rng(seed)
        self.W = rs.randn(n_actions, dim) * 0.05
        self.n_actions = n_actions
        self.dim = dim

    def logits(self, x):
        return x @ self.W.T

    def probs(self, x):
        logits = self.logits(x)
        logits -= logits.max(axis=-1, keepdims=True)
        exp_logits = np.exp(logits)
        return exp_logits / exp_logits.sum(axis=-1, keepdims=True)

    def predict(self, x):
        return np.argmax(self.probs(x), axis=-1)

    def copy(self):
        new_policy = SimpleLinearPolicy(self.dim, self.n_actions)
        new_policy.W = self.W.copy()
        return new_policy


# ─────────────────────────────────────────────────────────────
# Section B: Data generation and training
# ─────────────────────────────────────────────────────────────

def make_sleeper_data(n_benign=400, n_trigger=100, dim=8, seed=0):
    """Generate labelled data for a sleeper-agent persistence simulation."""
    rs = rng(seed + 4000)

    X_benign = rs.randn(n_benign, dim)
    X_benign[:, 0] += 1.5

    X_trigger = rs.randn(n_trigger, dim)
    X_trigger[:, 1] += 2.5

    return {
        "X_benign": X_benign,
        "y_benign": np.zeros(n_benign, dtype=int),
        "X_trigger": X_trigger,
        "y_trigger_pretrain": 2 * np.ones(n_trigger, dtype=int),
    }


def cross_entropy_step(policy, X, y, lr=0.05):
    """One full-batch cross-entropy gradient step."""
    probs = policy.probs(X)
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), y] = 1.0
    grad_W = ((probs - one_hot).T @ X) / len(y)
    policy.W -= lr * grad_W
    return policy


def pretrain_policy(data, n_epochs=150, lr=0.08, seed=0):
    """Pre-train on benign examples plus trigger-conditioned harmful behaviour."""
    policy = SimpleLinearPolicy(dim=data["X_benign"].shape[1], seed=seed)
    X_all = np.vstack([data["X_benign"], data["X_trigger"]])
    y_all = np.hstack([data["y_benign"], data["y_trigger_pretrain"]])
    for _ in range(n_epochs):
        policy = cross_entropy_step(policy, X_all, y_all, lr=lr)
    return policy


def sft_safety_tune(policy, data, n_epochs=120, lr=0.04):
    """Fine-tune on benign-only data to mimic standard safety tuning."""
    for _ in range(n_epochs):
        policy = cross_entropy_step(policy, data["X_benign"], data["y_benign"], lr=lr)
    return policy


def evaluate_sleeper(policy, data):
    """Return benign safety and trigger-conditioned attack success."""
    benign_preds = policy.predict(data["X_benign"])
    trigger_preds = policy.predict(data["X_trigger"])
    benign_safety = (benign_preds == 0).mean()
    attack_success = (trigger_preds == 2).mean()
    return float(benign_safety), float(attack_success)


# ─────────────────────────────────────────────────────────────
# Section B: Validation-calibrated BEAT-inspired detector
# ─────────────────────────────────────────────────────────────

def calibrated_beat_detector(policy, data, probe_strength=0.9, validation_frac=0.4, seed=0):
    """Calibrate a probe threshold on a held-out validation split, then test it."""
    probe = np.zeros(data["X_benign"].shape[1])
    probe[:3] = probe_strength

    def jsd_scores(X):
        p_orig = policy.probs(X)
        p_probe = policy.probs(X + probe[None, :])
        return np.array([
            jensen_shannon_divergence(p_orig[i], p_probe[i])
            for i in range(len(X))
        ])

    jsd_benign = jsd_scores(data["X_benign"])
    jsd_trigger = jsd_scores(data["X_trigger"])

    rs = rng(seed + 9100)
    benign_perm = rs.permutation(len(jsd_benign))
    trigger_perm = rs.permutation(len(jsd_trigger))

    benign_cut = max(1, int(len(jsd_benign) * validation_frac))
    trigger_cut = max(1, int(len(jsd_trigger) * validation_frac))

    benign_val = jsd_benign[benign_perm[:benign_cut]]
    benign_test = jsd_benign[benign_perm[benign_cut:]]
    trigger_val = jsd_trigger[trigger_perm[:trigger_cut]]
    trigger_test = jsd_trigger[trigger_perm[trigger_cut:]]

    candidate_thresholds = np.quantile(
        np.concatenate([benign_val, trigger_val]),
        np.linspace(0.05, 0.95, 60)
    )

    best_threshold = None
    best_balanced_acc = -1.0
    for threshold in candidate_thresholds:
        tpr = (trigger_val > threshold).mean()
        tnr = (benign_val <= threshold).mean()
        balanced_acc = 0.5 * (tpr + tnr)
        if balanced_acc > best_balanced_acc:
            best_balanced_acc = balanced_acc
            best_threshold = float(threshold)

    m = min(len(benign_test), len(trigger_test))
    benign_eval = benign_test[:m]
    trigger_eval = trigger_test[:m]
    true_labels = np.array([0] * m + [1] * m)
    pred_labels = (np.concatenate([benign_eval, trigger_eval]) > best_threshold).astype(int)

    detection_acc = float((true_labels == pred_labels).mean())
    auroc = binary_auroc(trigger_test, benign_test)
    test_tpr = float((trigger_eval > best_threshold).mean())
    test_fpr = float((benign_eval > best_threshold).mean())

    return {
        "threshold": best_threshold,
        "detection_acc": detection_acc,
        "auroc": auroc,
        "test_tpr": test_tpr,
        "test_fpr": test_fpr,
        "jsd_benign_all": jsd_benign,
        "jsd_trigger_all": jsd_trigger,
    }


# ─────────────────────────────────────────────────────────────
# Section B: Run simulation
# ─────────────────────────────────────────────────────────────

print("Running Section B sleeper-agent simulation...")

rows_stage = []
rows_det = []
example_probe = None

for seed in range(6):
    data = make_sleeper_data(seed=seed)
    pol_pre = pretrain_policy(data, seed=seed)
    pre_ben, pre_atk = evaluate_sleeper(pol_pre, data)

    pol_sft = sft_safety_tune(pol_pre.copy(), data, n_epochs=120, lr=0.04)
    sft_ben, sft_atk = evaluate_sleeper(pol_sft, data)

    pol_sft_plus = sft_safety_tune(pol_pre.copy(), data, n_epochs=300, lr=0.06)
    sftp_ben, sftp_atk = evaluate_sleeper(pol_sft_plus, data)

    rows_stage.extend([
        {"seed": seed, "stage": "Pre-training", "benign_safety": pre_ben, "attack_success": pre_atk},
        {"seed": seed, "stage": "After SFT", "benign_safety": sft_ben, "attack_success": sft_atk},
        {"seed": seed, "stage": "After SFT+", "benign_safety": sftp_ben, "attack_success": sftp_atk},
    ])

    det = calibrated_beat_detector(pol_sft, data, seed=seed)
    rows_det.append({
        "seed": seed,
        "threshold": det["threshold"],
        "detection_acc": det["detection_acc"],
        "auroc": det["auroc"],
        "test_tpr": det["test_tpr"],
        "test_fpr": det["test_fpr"],
    })

    if seed == 0:
        example_probe = det

df_b = pd.DataFrame(rows_stage)
df_b_det = pd.DataFrame(rows_det)

print("  ✓ Validation-calibrated probe metrics aggregated across seeds.")
print(f"  ✓ Mean probe AUROC: {df_b_det['auroc'].mean():.2%}")

# ─────────────────────────────────────────────────────────────
# Section B: Plots
# ─────────────────────────────────────────────────────────────

stages = ["Pre-training", "After SFT", "After SFT+"]
ben_vals = [df_b[df_b.stage == s]["benign_safety"].mean() for s in stages]
atk_vals = [df_b[df_b.stage == s]["attack_success"].mean() for s in stages]
ben_stds = [df_b[df_b.stage == s]["benign_safety"].std() for s in stages]
atk_stds = [df_b[df_b.stage == s]["attack_success"].std() for s in stages]

fig, axes = plt.subplots(1, 3, figsize=(15.6, 4.8))
fig.suptitle(
    "Section B — Sleeper-Agent Persistence and Validation-Calibrated Probing\n"
    "(Capability (a): pre-training data access)",
    fontsize=11
)

x = np.arange(len(stages))
w = 0.35

ax = axes[0]
ax.bar(x - w / 2, ben_vals, w, color=COLORS["clean"], alpha=0.85,
       yerr=ben_stds, capsize=4, label="Benign Safety")
ax.bar(x + w / 2, atk_vals, w, color=COLORS["attack"], alpha=0.85,
       yerr=atk_stds, capsize=4, label="Attack Success")
ax.set_xticks(x)
ax.set_xticklabels(stages, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Rate")
ax.set_ylim(0, 1.1)
ax.set_title("Safety vs Trigger-Attack Across Stages")
ax.axhline(0.5, color="grey", lw=0.7, linestyle="--", alpha=0.5)
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
ax.legend(frameon=True)

ax = axes[1]
for seed in range(6):
    seed_rows = df_b[df_b.seed == seed]
    ben_seed = [seed_rows[seed_rows.stage == s]["benign_safety"].values[0] for s in stages]
    atk_seed = [seed_rows[seed_rows.stage == s]["attack_success"].values[0] for s in stages]
    ax.plot([0, 1, 2], ben_seed, color=COLORS["clean"], alpha=0.35, linewidth=1)
    ax.plot([0, 1, 2], atk_seed, color=COLORS["attack"], alpha=0.35, linewidth=1)
ax.plot([0, 1, 2], ben_vals, color=COLORS["clean"], linewidth=2.5, marker="o",
        label="Benign safety (mean)")
ax.plot([0, 1, 2], atk_vals, color=COLORS["attack"], linewidth=2.5, marker="s",
        label="Attack success (mean)")
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(stages, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Rate")
ax.set_ylim(0, 1.1)
ax.set_title("Per-Seed Trajectories")
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
ax.legend(fontsize=8, frameon=True)

ax = axes[2]
jsd_b = example_probe["jsd_benign_all"]
jsd_t = example_probe["jsd_trigger_all"]
bins = np.linspace(0, max(jsd_b.max(), jsd_t.max()) * 1.10, 28)
ax.hist(jsd_b, bins=bins, color=COLORS["clean"], alpha=0.6, label="Benign examples")
ax.hist(jsd_t, bins=bins, color=COLORS["attack"], alpha=0.6, label="Triggered examples")
ax.axvline(example_probe["threshold"], color="black", linestyle="--", linewidth=1.2,
           label="Validation-calibrated threshold")
ax.set_xlabel("JSD (probe-response distribution shift)")
ax.set_ylabel("Count")
ax.set_title(
    "Probe Score Distributions (seed 0)\n"
    f"Mean test acc={df_b_det['detection_acc'].mean():.2%}, mean AUROC={df_b_det['auroc'].mean():.2%}"
)
ax.legend(fontsize=8, frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

stage_summary = (
    df_b.groupby("stage")[["benign_safety", "attack_success"]]
       .mean()
       .reindex(stages)
       .reset_index()
       .round(3)
)
stage_summary.columns = ["Stage", "Benign safety", "Attack success"]

detector_summary = df_b_det[["seed", "threshold", "detection_acc", "auroc", "test_tpr", "test_fpr"]].copy()
detector_summary = detector_summary.round(3)
detector_summary.columns = ["Seed", "Threshold", "Accuracy", "AUROC", "TPR", "FPR"]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.2))
render_table(
    axes[0],
    stage_summary,
    title="Stage summary",
    col_widths=[0.42, 0.29, 0.29],
    font_size=8.6,
    cell_height=0.22,
)
render_table(
    axes[1],
    detector_summary,
    title="Probe summary across seeds",
    col_widths=[0.10, 0.20, 0.18, 0.17, 0.17, 0.18],
    font_size=8.1,
    cell_height=0.18,
)
plt.tight_layout()
plt.show()



### Section B — Interpretation

**What the plots show:**
- After pre-training, the policy strongly executes the harmful action on triggered examples.
- Standard benign-only safety tuning improves benign behaviour, but the trigger-conditioned pathway still survives.
- Stronger SFT+ reduces the attack further, yet it still does not fully eliminate the sleeper mechanism in this surrogate.
- The probe detector now uses a **held-out validation split** to set its threshold and reports **mean test metrics across seeds** rather than a single fixed cutoff.

**What the calibrated probe result means:**
The revised detector typically achieves **moderate**, not perfect, separation. That is a more realistic outcome for this notebook: probing helps, but it does not cleanly separate triggered and benign behaviour in every run.

**How this relates to BEAT in the paper:**
The paper cites BEAT's very high AUROC only for **standard, non-adaptive** backdoor settings. The more modest probe numbers in this notebook should be read as a **probe-aware / validation-calibrated surrogate outcome**, not as a reproduction of BEAT's headline result.

**Why this matters for the paper:**
The persistence claim still holds. Safety tuning on benign data improves ordinary behaviour more than it suppresses rare trigger-conditioned behaviour, which is exactly the concealment-versus-elimination distinction emphasized in the sleeper-agent literature.

> ⚠️ **Caveats:** The detector is still a black-box surrogate and the policy is still linear. Real sleeper attacks can adapt to probing, and larger models may conceal trigger behaviour differently.


---
## Section C: Indirect Prompt Injection and MCP Tool-Schema Poisoning

### Paper context (§2.4, §2.4.1)

The paper identifies a fundamental architectural gap: LLMs lack built-in separation between trusted system instructions and untrusted external content.

- **IPI** [3]: attackers embed instructions in retrieved content; the model follows them as trusted directives.
- **MCP tool-schema poisoning** [17]: MCPTox reports up to **72.8% attack success** on o1-mini with <3% refusal.
- **FuncPoison** [18]: poisoning tool *metadata alone* (no training-data access) is sufficient to hijack agent behaviour.

Attack variants from the paper:
| Variant | Description |
|---------|-------------|
| *Shadowing* | Contaminate shared model context via a trusted tool |
| *Rug Pulls* | Benign at approval time, malicious post-deployment |
| *Full-Schema Poisoning* | Exploit the entire tool definition, not just one field |

> Capability tier: **(c)** — deployment-time retrieval / tool-schema access

### ⚠️ What this simulation does NOT prove
- OS-level command injection (CVE-2025-6514) or remote code execution (CVE-2025-49596) — these are infrastructure-layer threats.
- Exact behaviour of any specific agent or MCP server.
- Defense effectiveness beyond the trust-weight assumptions used here.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Section C: Multi-logit action selector
# ─────────────────────────────────────────────────────────────

def softmax_logits(logits):
    logits = np.asarray(logits, dtype=float)
    logits = logits - logits.max()
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum()


def action_selector(config, malicious_retrieval=0.0, malicious_tool=0.0,
                    system_val=1.0, user_val=0.40,
                    benign_retrieval=0.22, benign_tool=0.18,
                    noise_std=0.32, seed=0):
    """Choose among safe / neutral / unsafe actions using source-specific logits."""
    rs = rng(seed)

    safe_logit = (
        1.65 * config["system_safe"] * system_val
        + 0.25 * user_val
        + 0.22 * benign_retrieval
        + 0.18 * benign_tool
        - 0.10 * (0.8 * malicious_retrieval + 0.8 * malicious_tool)
    )

    neutral_logit = 1.00 + 0.12 * user_val + 0.15 * (benign_retrieval + benign_tool)

    unsafe_logit = (
        0.70
        + config["guard"] * (
            1.55 * config["ret_scale"] * malicious_retrieval
            + 1.45 * config["tool_scale"] * malicious_tool
        )
        + 0.10 * user_val
        - 0.22 * config["system_safe"] * system_val
    )

    logits = np.array([safe_logit, neutral_logit, unsafe_logit]) + rs.randn(3) * noise_std
    probs = softmax_logits(logits)
    action_idx = int(np.argmax(probs))
    action = ["safe", "neutral", "unsafe"][action_idx]
    return action, probs


def evaluate_attack_case(configs, malicious_retrieval, malicious_tool,
                         n_trials=250, n_seeds=6):
    """Average unsafe-action rates across defense configs and random seeds."""
    raw = {name: [] for name in configs}

    for seed in range(n_seeds):
        for name, config in configs.items():
            unsafe_hits = 0
            for trial in range(n_trials):
                action, _ = action_selector(
                    config,
                    malicious_retrieval=malicious_retrieval,
                    malicious_tool=malicious_tool,
                    seed=seed * 10000 + trial,
                )
                unsafe_hits += int(action == "unsafe")
            raw[name].append(unsafe_hits / n_trials)

    return {
        name: {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        for name, vals in raw.items()
    }


DEFENSE_CONFIGS = {
    "no_defense":          {"system_safe": 1.00, "ret_scale": 1.00, "tool_scale": 1.00, "guard": 1.00},
    "instruction_sep":     {"system_safe": 1.10, "ret_scale": 0.35, "tool_scale": 1.00, "guard": 0.96},
    "metadata_validation": {"system_safe": 1.08, "ret_scale": 1.00, "tool_scale": 0.35, "guard": 0.96},
    "scoped_execution":    {"system_safe": 1.12, "ret_scale": 0.60, "tool_scale": 0.60, "guard": 0.82},
    "combined":            {"system_safe": 1.12, "ret_scale": 0.50, "tool_scale": 0.50, "guard": 0.82},
}

ATTACK_CASES = {
    "combined": {"malicious_retrieval": 0.70, "malicious_tool": 0.60},
    "ipi":      {"malicious_retrieval": 1.25, "malicious_tool": 0.00},
    "tool":     {"malicious_retrieval": 0.00, "malicious_tool": 1.10},
}

print("Running Section C with source-specific safe/neutral/unsafe logits...")

combined_results = evaluate_attack_case(DEFENSE_CONFIGS, **ATTACK_CASES["combined"])
ipi_results = evaluate_attack_case(DEFENSE_CONFIGS, **ATTACK_CASES["ipi"])
tool_results = evaluate_attack_case(DEFENSE_CONFIGS, **ATTACK_CASES["tool"])

print("  ✓ Section C computations complete.")

# ─────────────────────────────────────────────────────────────
# Section C: Plots
# ─────────────────────────────────────────────────────────────

DEFENSE_LABELS = ["No Defense", "Instr. Sep.", "Meta Valid.", "Scoped Exec.", "Combined"]
DEFENSE_KEYS = list(DEFENSE_CONFIGS.keys())
BAR_COLORS = [COLORS["nodef"], COLORS["ppo"], COLORS["clean"], COLORS["pretrain"], COLORS["def"]]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.1))
fig.suptitle(
    "Section C — IPI and MCP Tool-Schema Poisoning\n"
    "(Source-specific unsafe logits; action-selection layer only)",
    fontsize=11
)

ax = axes[0]
x_left = np.arange(len(DEFENSE_KEYS))
combined_means = [combined_results[k]["mean"] for k in DEFENSE_KEYS]
combined_stds = [combined_results[k]["std"] for k in DEFENSE_KEYS]
bars = ax.bar(x_left, combined_means, color=BAR_COLORS, alpha=0.85,
              yerr=combined_stds, capsize=4, edgecolor="white", linewidth=0.8)
ax.set_ylabel("Unsafe Action Rate")
ax.set_title("Combined Retrieval + Tool Poisoning")
ax.set_ylim(0, 1.05)
ax.set_xticks(x_left)
ax.set_xticklabels(DEFENSE_LABELS, rotation=22, ha="right", fontsize=9)
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
for bar, val in zip(bars, combined_means):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.2f}", ha="center", fontsize=8)

ax = axes[1]
x = np.arange(len(DEFENSE_KEYS))
width = 0.35
ipi_means = [ipi_results[k]["mean"] for k in DEFENSE_KEYS]
ipi_stds = [ipi_results[k]["std"] for k in DEFENSE_KEYS]
tool_means = [tool_results[k]["mean"] for k in DEFENSE_KEYS]
tool_stds = [tool_results[k]["std"] for k in DEFENSE_KEYS]

ax.bar(x - width / 2, ipi_means, width, color=COLORS["attack"], alpha=0.85,
       yerr=ipi_stds, capsize=4, label="IPI only (retrieval malicious)", edgecolor="white")
ax.bar(x + width / 2, tool_means, width, color=COLORS["cascade"], alpha=0.85,
       yerr=tool_stds, capsize=4, label="Tool-schema only", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(DEFENSE_LABELS, rotation=22, ha="right", fontsize=9)
ax.set_ylabel("Unsafe Action Rate")
ax.set_title("Attack Vectors Separated by Source")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.18, linewidth=0.7)
ax.legend(fontsize=8, frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

df_c = pd.DataFrame({
    "Defense": DEFENSE_LABELS,
    "Combined": [round(combined_results[k]["mean"], 3) for k in DEFENSE_KEYS],
    "IPI only": [round(ipi_results[k]["mean"], 3) for k in DEFENSE_KEYS],
    "Tool schema": [round(tool_results[k]["mean"], 3) for k in DEFENSE_KEYS],
})

fig, ax = plt.subplots(figsize=(6.8, 2.5))
render_table(
    ax,
    df_c,
    title="Section C summary",
    col_widths=[0.28, 0.24, 0.24, 0.24],
    font_size=8.5,
    cell_height=0.19,
)
plt.tight_layout()
plt.show()



### Section C — Interpretation

**What changed in this revision:**
- The action model is no longer a single scalar that almost always collapses to "unsafe."
- It now uses separate **safe, neutral, and unsafe logits**, so different context sources can push behaviour in different directions.

**What the plots now show:**
- Without defenses, malicious retrieval and malicious tool metadata both drive high unsafe-action rates.
- **Instruction-data separation** sharply reduces the retrieval-only attack, but it is much less effective against tool poisoning.
- **Metadata validation** sharply reduces tool-schema poisoning, but not retrieval-only IPI.
- **Scoped execution** helps against both sources, and the **combined** configuration performs best overall.

**Why this matters:**
The revised plots now express the intended paper claim: indirect prompt injection and tool-schema poisoning are related trust-boundary failures, but they are **not the same failure mode** and should not respond identically to the same defense.

> ⚠️ **Caveats:** This remains a surrogate of the action-selection interface. It does not model OS-level exploitation, MCP infrastructure CVEs, or adaptive attacks that mimic benign trust signals.


---
## Section D: Multi-Agent Cascading Failure with α_observed and Layer-3 Circuit Breaker

### Paper context (§2.5, §5)

The paper formalises multi-agent contamination as an epidemiological process [4]:

$$\alpha_{\text{observed}}(t) = \frac{\text{anomalous\_count}(t+1)}{\max(\text{anomalous\_count}(t),\; 1)}$$

> ⚠️ **Critical distinction:**
> - **α_observed** = *runtime* metric tracked by the Layer-3 monitor (plotted in this section)
> - **α_effective** = *design-time* conceptual formula (Eq. 2 in paper): `α_eff = α_raw × ∏ P(Bᵢ|B₁:ᵢ₋₁)` — **not** the runtime plotted variable

### Layer-3 Circuit Breaker (LCD §5)

| State | Trigger | Response |
|-------|---------|----------|
| 🟡 Yellow | α_observed ≈ 0.9–1.0 | Increased logging only |
| 🟠 Orange | α_observed > 1.0 | Reduce connectivity / revoke broadcast privileges |
| 🔴 Red | α_observed ≥ 2.0 | Quarantine subgraphs / isolate contaminated nodes |

### Topology findings (§2.5, [19, 20])

- **Dense / SharedPool** topologies: high connectivity → rapid contamination spread
- **Sparse / decentralised** topologies: lower connectivity → inherent fault isolation

> Capability tier: **(d)** — participation in a multi-agent pipeline

### ⚠️ What this simulation does NOT prove
- That α_observed = 1 is a universal production quarantine threshold.
- That the circuit breaker validates the full LCD architecture.
- Contamination dynamics in real heterogeneous, cross-modal agent networks.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Section D: Graph construction
# ─────────────────────────────────────────────────────────────

def build_dense_graph(n_nodes=60, edge_prob=0.12, seed=0):
    """Erdős–Rényi graph used as a dense SharedPool-like topology."""
    return nx.erdos_renyi_graph(n=n_nodes, p=edge_prob, seed=seed)


def build_sparse_graph(n_nodes=60, m_edges=2, seed=0):
    """Barabási–Albert graph used as a more decentralised topology."""
    return nx.barabasi_albert_graph(n=n_nodes, m=m_edges, seed=seed)


# ─────────────────────────────────────────────────────────────
# Section D: Cascade simulation with quarantine-aware breaker
# ─────────────────────────────────────────────────────────────

def simulate_cascade(G, spread_prob=0.16, n_initial=2, n_steps=18,
                     seed=0, use_breaker=False,
                     yellow_thresh=1.00, orange_thresh=1.10, red_thresh=2.0,
                     red_quarantine_frac=0.35):
    """Simulate active contamination spread with an isolation-based breaker.

    Active contaminated nodes can spread. Quarantined nodes are isolated,
    remain affected, and cannot spread or be re-infected.
    """
    rs = rng(seed + 7000)
    nodes = list(G.nodes())
    contaminated = set(rs.choice(nodes, size=n_initial, replace=False).tolist())
    quarantined = set()

    active_counts = [len(contaminated)]
    affected_counts = [len(contaminated)]
    quarantined_counts = [0]
    alpha_over_time = [1.0]
    breaker_states = ["green"]
    current_spread_prob = spread_prob

    for _ in range(n_steps - 1):
        prev_active = max(len(contaminated), 1)
        new_contaminated = set()

        for node in list(contaminated):
            for neighbor in G.neighbors(node):
                if neighbor in contaminated or neighbor in quarantined:
                    continue
                if rs.rand() < current_spread_prob:
                    new_contaminated.add(neighbor)

        contaminated |= new_contaminated
        raw_alpha = len(contaminated) / prev_active
        state = "green"

        if use_breaker:
            if raw_alpha >= red_thresh and contaminated:
                state = "red"
                n_quarantine = max(1, int(np.ceil(red_quarantine_frac * len(contaminated))))
                ranked_nodes = sorted(
                    contaminated,
                    key=lambda node: (G.degree(node), rs.rand()),
                    reverse=True,
                )
                for node in ranked_nodes[:n_quarantine]:
                    contaminated.discard(node)
                    quarantined.add(node)
                current_spread_prob = spread_prob * 0.28
            elif raw_alpha >= orange_thresh:
                state = "orange"
                current_spread_prob = spread_prob * 0.55
            elif raw_alpha >= yellow_thresh:
                state = "yellow"
                current_spread_prob = spread_prob * 0.78
            else:
                current_spread_prob = spread_prob

        active_counts.append(len(contaminated))
        affected_counts.append(len(contaminated) + len(quarantined))
        quarantined_counts.append(len(quarantined))
        alpha_over_time.append(raw_alpha)
        breaker_states.append(state)

    return {
        "active": active_counts,
        "affected": affected_counts,
        "quarantined": quarantined_counts,
        "alpha": alpha_over_time,
        "states": breaker_states,
    }


# ─────────────────────────────────────────────────────────────
# Section D: Run all four conditions × multiple seeds
# ─────────────────────────────────────────────────────────────

print("Running Section D multi-agent cascade simulation...")

N_NODES = 60
N_STEPS = 18
N_SEEDS = 8
N_INITIAL = 2
SPREAD = 0.16

CONDITIONS = {
    "dense_no_breaker": {"graph_fn": build_dense_graph, "breaker": False},
    "dense_with_breaker": {"graph_fn": build_dense_graph, "breaker": True},
    "sparse_no_breaker": {"graph_fn": build_sparse_graph, "breaker": False},
    "sparse_with_breaker": {"graph_fn": build_sparse_graph, "breaker": True},
}

raw = {cn: {"active": [], "affected": [], "quarantined": [], "alpha": [], "states": []}
       for cn in CONDITIONS}

for seed in range(N_SEEDS):
    for cn, cond in CONDITIONS.items():
        G = cond["graph_fn"](n_nodes=N_NODES, seed=seed)
        rec = simulate_cascade(
            G,
            spread_prob=SPREAD,
            n_initial=N_INITIAL,
            n_steps=N_STEPS,
            seed=seed,
            use_breaker=cond["breaker"],
        )
        raw[cn]["active"].append(rec["active"])
        raw[cn]["affected"].append(rec["affected"])
        raw[cn]["quarantined"].append(rec["quarantined"])
        raw[cn]["alpha"].append(rec["alpha"])
        raw[cn]["states"].append(rec["states"])


D_SUMM = {}
metric_rows = []
for cn, rec in raw.items():
    active_arr = np.array(rec["active"])
    affected_arr = np.array(rec["affected"])
    quarantined_arr = np.array(rec["quarantined"])
    alpha_arr = np.array(rec["alpha"])

    time25 = [first_reach_time(series, 0.25 * N_NODES) for series in active_arr]
    time50 = [first_reach_time(series, 0.50 * N_NODES) for series in active_arr]
    auc_vals = [normalised_auc(series, N_NODES) for series in active_arr]
    peak_alpha_vals = [float(np.max(series[1:])) for series in alpha_arr]

    D_SUMM[cn] = {
        "active_mean": active_arr.mean(axis=0),
        "active_std": active_arr.std(axis=0),
        "quarantined_mean": quarantined_arr.mean(axis=0),
        "quarantined_std": quarantined_arr.std(axis=0),
        "alpha_mean": alpha_arr.mean(axis=0),
        "alpha_std": alpha_arr.std(axis=0),
        "final_active_mean": float(active_arr[:, -1].mean()),
        "final_active_std": float(active_arr[:, -1].std()),
        "final_affected_mean": float(affected_arr[:, -1].mean()),
        "time25_mean": float(np.nanmean(time25)),
        "time25_std": float(np.nanstd(time25)),
        "time50_mean": float(np.nanmean(time50)),
        "time50_std": float(np.nanstd(time50)),
        "auc_mean": float(np.mean(auc_vals)),
        "auc_std": float(np.std(auc_vals)),
        "peak_alpha_mean": float(np.mean(peak_alpha_vals)),
        "peak_alpha_std": float(np.std(peak_alpha_vals)),
    }

    metric_rows.append({
        "Condition": cn,
        "FinalActive": round(D_SUMM[cn]["final_active_mean"], 2),
        "FinalAffected": round(D_SUMM[cn]["final_affected_mean"], 2),
        "T25": round(D_SUMM[cn]["time25_mean"], 2),
        "T50": round(D_SUMM[cn]["time50_mean"], 2),
        "NormAUC": round(D_SUMM[cn]["auc_mean"], 3),
        "PeakAlpha": round(D_SUMM[cn]["peak_alpha_mean"], 3),
    })

df_d_metrics = pd.DataFrame(metric_rows)
print("  Operational cascade summary:")
print(df_d_metrics.to_string(index=False))

# ─────────────────────────────────────────────────────────────
# Section D: Plots
# ─────────────────────────────────────────────────────────────

T = np.arange(N_STEPS)
C_MAP = {
    "dense_no_breaker": (COLORS["nodef"], "--", "Dense — No Breaker"),
    "dense_with_breaker": (COLORS["breaker"], "-", "Dense — With Breaker"),
    "sparse_no_breaker": (COLORS["cascade"], "--", "Sparse — No Breaker"),
    "sparse_with_breaker": (COLORS["def"], "-", "Sparse — With Breaker"),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Section D — Multi-Agent Cascading Failure with Quarantine-Aware Circuit Breaking\n"
    "(Capability (d): participation in multi-agent pipeline)",
    fontsize=11
)

ax = axes[0, 0]
for cn, (col, ls, lbl) in C_MAP.items():
    s = D_SUMM[cn]
    plot_line_with_band(ax, T, s["active_mean"], s["active_std"], col, lbl)
    ax.plot(T, s["active_mean"], color=col, linestyle=ls, linewidth=2)
ax.set_xlabel("Time Step")
ax.set_ylabel("Active Contaminated Agent Count")
ax.set_title("Active Contamination Over Time")
ax.set_ylim(0, N_NODES + 5)
ax.legend(fontsize=8)

ax = axes[0, 1]
for cn in ["dense_no_breaker", "dense_with_breaker"]:
    col, ls, lbl = C_MAP[cn]
    s = D_SUMM[cn]
    plot_line_with_band(ax, T, s["alpha_mean"], s["alpha_std"], col, lbl)
    ax.plot(T, s["alpha_mean"], color=col, linestyle=ls, linewidth=2)
ax.axhline(1.00, color="goldenrod", linewidth=1.0, linestyle=":", label="Yellow: α ≈ 1.00")
ax.axhline(1.10, color="darkorange", linewidth=1.0, linestyle=":", label="Orange: α > 1.10")
ax.axhline(2.00, color="red", linewidth=1.0, linestyle=":", label="Red: α ≥ 2.00")
ax.set_xlabel("Time Step")
ax.set_ylabel("α_observed(t)")
ax.set_title("α_observed Over Time — Dense Topology")
ax.set_ylim(0, 3.2)
ax.legend(fontsize=7)

ax = axes[1, 0]
x = np.arange(len(C_MAP))
width = 0.35
time25_vals = [D_SUMM[cn]["time25_mean"] for cn in C_MAP]
time25_stds = [D_SUMM[cn]["time25_std"] for cn in C_MAP]
time50_vals = [D_SUMM[cn]["time50_mean"] for cn in C_MAP]
time50_stds = [D_SUMM[cn]["time50_std"] for cn in C_MAP]
ax.bar(x - width / 2, time25_vals, width, color="#74C69D", yerr=time25_stds,
       capsize=4, edgecolor="white", label="Time to 25% contamination")
ax.bar(x + width / 2, time50_vals, width, color="#1B4332", yerr=time50_stds,
       capsize=4, edgecolor="white", label="Time to 50% contamination")
ax.set_xticks(x)
ax.set_xticklabels([C_MAP[cn][2] for cn in C_MAP], rotation=18, ha="right", fontsize=8)
ax.set_ylabel("Time Step")
ax.set_title("Threshold-Crossing Delay")
ax.legend(fontsize=8)

ax = axes[1, 1]
auc_vals = [D_SUMM[cn]["auc_mean"] for cn in C_MAP]
auc_stds = [D_SUMM[cn]["auc_std"] for cn in C_MAP]
peak_vals = [D_SUMM[cn]["peak_alpha_mean"] for cn in C_MAP]
peak_stds = [D_SUMM[cn]["peak_alpha_std"] for cn in C_MAP]
ax.bar(x - width / 2, auc_vals, width, color="#4EA8DE", yerr=auc_stds,
       capsize=4, edgecolor="white", label="Normalised exposure AUC")
ax.set_ylabel("Normalised Exposure AUC")
ax.set_ylim(0, 1.0)
ax.set_xticks(x)
ax.set_xticklabels([C_MAP[cn][2] for cn in C_MAP], rotation=18, ha="right", fontsize=8)

ax2 = ax.twinx()
ax2.bar(x + width / 2, peak_vals, width, color="#6C757D", alpha=0.55,
        yerr=peak_stds, capsize=4, edgecolor="white", label="Peak α_observed")
ax2.set_ylabel("Peak α_observed")
ax2.set_ylim(0, 3.2)
ax.set_title("Exposure vs Peak Growth")

handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax.legend(handles1 + handles2, labels1 + labels2, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

print("\nSection D complete.")


### Section D — Interpretation

**What changed in this revision:**
- The red-tier breaker no longer "cures" nodes by deleting them from the contaminated set.
- It now creates a **quarantined state**: isolated nodes stay affected, but they stop propagating contamination.
- The section now reports **operational spread metrics** such as time-to-25%, time-to-50%, and exposure AUC.
- The plotted threshold bands now track the paper more closely: **yellow ≈ 1**, **orange > 1**, and **red ≥ 2**.

**What the plots now show:**
1. **Dense topologies spread faster** than sparse ones when no breaker is present.
2. The breaker delays threshold crossings in both topologies, especially in the dense case.
3. Exposure AUC drops substantially when the breaker is enabled, meaning the system spends less time with a large active contaminated population.
4. Final active contamination also falls in this setup, but the strongest effect is on **cascade speed and exposure**, not a claim of universal eradication.

**How to read α_observed now:**
α_observed is computed on the **active propagating set**. Quarantined nodes are tracked separately, because they remain affected but are no longer able to spread contamination. The orange line is drawn slightly above 1.0 so the paper's strict **α > 1** condition is legible in a static figure.

> ⚠️ **Caveats:** Breaker thresholds, quarantine fraction, and graph family are all simulation parameters. This section illustrates kinetic control, not a validated enterprise breaker policy.


---
## Final Summary Table

Mapping each simulation to its paper taxonomy dimensions.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Final summary table — maps each simulation to paper taxonomy
# ─────────────────────────────────────────────────────────────

import textwrap

summary_rows = [
    {
        "Simulation": "A — DPO vs PPO",
        "Attack Class": "Preference-data poisoning [6, 8]",
        "Pipeline Stage": "Alignment",
        "Capability Tier": "(b) preference-data",
        "Core Metric": "Held-out attack success vs low-end overall poison rate",
        "Main Observation": "Low-end overall-dataset poisoning lifts DPO attack success earlier than PPO in this surrogate.",
        "Relevant Defense": "Reward buffering; preference-data auditing",
        "Main Caveat": "Illustrative surrogate only; does not numerically reproduce the 0.5% threshold from [6].",
    },
    {
        "Simulation": "B — Sleeper Persistence",
        "Attack Class": "Sleeper agent / persistent pre-training poison [11, 12]",
        "Pipeline Stage": "Pre-training → Alignment",
        "Capability Tier": "(a) pre-training data",
        "Core Metric": "Benign safety / attack success / validation-calibrated probe AUROC",
        "Main Observation": "Safety tuning improves benign behaviour more than it removes trigger-conditioned behaviour.",
        "Relevant Defense": "Validation-calibrated probing [24]; pre-deployment revocation [25]",
        "Main Caveat": "Probe separation is moderate by design here and should not be read as BEAT's non-adaptive baseline.",
    },
    {
        "Simulation": "C — IPI & MCP",
        "Attack Class": "Indirect prompt injection [3]; MCP tool-schema poisoning [17, 18]",
        "Pipeline Stage": "Deployment",
        "Capability Tier": "(c) retrieval/tool-schema",
        "Core Metric": "Unsafe-action rate per defense config",
        "Main Observation": "Instruction separation helps IPI most; metadata validation helps tool poisoning most.",
        "Relevant Defense": "Instruction-data separation; metadata validation; scoped execution",
        "Main Caveat": "Action-selection layer only. Not an infrastructure exploit model.",
    },
    {
        "Simulation": "D — Multi-Agent Cascade",
        "Attack Class": "Multi-agent cascading failure [4, 20]",
        "Pipeline Stage": "Multi-agent execution",
        "Capability Tier": "(d) multi-agent pipeline",
        "Core Metric": "Time-to-threshold, exposure AUC, peak α_observed",
        "Main Observation": "Quarantine-aware breaker mainly slows spread and lowers exposure, while often reducing final active cascade size.",
        "Relevant Defense": "Topology-aware monitoring; Layer-3 LCD breaker",
        "Main Caveat": "Thresholds and quarantine policy are scenario-specific calibration choices.",
    },
]

df_final = pd.DataFrame(summary_rows)

display_cols = ["Simulation", "Pipeline Stage", "Capability Tier",
                "Core Metric", "Relevant Defense", "Main Caveat"]
final_display = df_final[display_cols].copy()

fig, ax = plt.subplots(figsize=(19.5, 7.0))
fig.suptitle("Simulation → Paper Taxonomy Mapping", fontsize=13, fontweight="bold", y=0.98)
render_table(
    ax,
    final_display,
    col_widths=[0.12, 0.12, 0.11, 0.19, 0.21, 0.25],
    wrap_widths={
        "Simulation": 20,
        "Pipeline Stage": 20,
        "Capability Tier": 20,
        "Core Metric": 34,
        "Relevant Defense": 38,
        "Main Caveat": 40,
    },
    font_size=8.2,
    cell_height=0.18,
    bbox=[0, 0.05, 1, 0.87],
)
fig.text(
    0.01,
    0.01,
    "Each row maps one simulation to the paper taxonomy slot it is intended to illustrate.",
    fontsize=9,
    color="#4A5568",
)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()



---
## Final Conclusion

### What This Revised Notebook Does

This notebook remains an **illustrative, literature-grounded simulation benchmark** supporting the paper *"The Privilege Inversion Pattern: A Conceptual Cross-Stage Framework for Data Contamination in Agentic AI Architectures"*.

The revised version now ties each section more tightly to the mechanism it is meant to illustrate:

| Section | Revised Mechanism Demonstrated |
|---------|--------------------------------|
| **A** | Low-end overall-dataset preference poisoning with held-out DPO vs PPO evaluation |
| **B** | Sleeper persistence through safety tuning with validation-calibrated probing |
| **C** | Source-specific trust-boundary failures for retrieved content vs tool metadata |
| **D** | Quarantine-aware cascade control measured through threshold delay, exposure, and paper-aligned α_observed states |

---

### What This Revised Notebook Still Does NOT Prove

1. **Universal thresholds.** No poisoning rate, α_observed threshold, or unsafe-action rate shown here should be reused as a deployment standard.

2. **Validation of the full LCD architecture.** The notebook illustrates design ideas consistent with LCD, but it does not evaluate a deployed defense stack.

3. **Behaviour of real LLMs or real agent frameworks.** All sections use lightweight surrogates with synthetic feature vectors or abstract network dynamics.

4. **Completeness of the threat taxonomy.** Several attack classes discussed in the paper remain out of scope for this notebook, including recursive collapse, multimodal contamination, memory poisoning, federated instruction tuning attacks, and benchmark leakage.

---

### Why the Revisions Matter

1. **Section A** now sweeps low-end contamination rates over the **overall** preference dataset, which is the regime actually relevant to the paper's cited 0.5% DPO fragility discussion.
2. **Section B** explicitly treats the probe result as a validation-calibrated, probe-aware surrogate rather than as a reproduction of BEAT's non-adaptive AUROC headline.
3. **Section C** still differentiates retrieved-content and tool-metadata attacks at the action-selection level instead of collapsing them into a single scalar score.
4. **Section D** now aligns its visual thresholds more closely with the paper's Layer-3 logic: yellow ≈ 1, orange > 1, red ≥ 2.
5. **The final mapping table** is now wrapped and resized so the notebook exports cleanly without clipped text.

---

> **This notebook is best understood as an illustrative benchmark for mechanisms and comparative trends, not a substitute for empirical evaluation on production models or real agentic systems.**

*All code is reproducible with `GLOBAL_SEED = 42`.*
